# 3.12/3.17 Pointing -- Update VIS Pointing Loop

OVERVIEW: Update the VIS pointing parameters in the control loop. Observe three stars in the same FOV with moderate brightness to quantify the pointing performance with new parameters in arcsecs/time. A long duration observation enables measurement during possible thermal variations over ~1 orbit: 100 min / 6000 sec. 


CHECK: The data should measure the pointing accuracy (absolute) and precision (rms variation from nominal) in arcseconds over time. 


EXIT CRITERIA: observation obtained, analysis complete and summarized in 3.12/3.17_ResultsSummary.md
- pointing accuracy required = ?
- precision required = ?


DATA: three 200x200 regions of interest (ROIs) on VISDA, one long-duration observation
- Note - data volume may be reduced by reducing temporal resolution
    
    
TARGET CRITERIA: 3 stars in the same VIDSA FOV, One in the center and two others at least 1/3 from the center in different directions from the center. Moderately bright targets with Pandora Mag 8-9 (approx Gaia B Mag). This magnitude criteria is flexible and currently being refined -- our science target list is selected for Jmag=7-11.5 and Hmag<11.


KEY STAKEHOLDERS: Tom Barclay

### Load in

In [ ]:
''' Make sure you have pandorasim installed and updated '''

import pandorasat as ps
import pandorasim as pp
ps.utils.get_phoenix_model(teff=7000, jmag=10) ### this is a temporary fix to an import error with pandorasim
from pandorasim import VisibleSim, NIRSim
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord, Longitude
from astropy.io import fits
from pandorasat.plotting import animate
from pandorasat.plotting import save_mp4, save_gif
from astropy.time import Time, TimeDelta

### Importing additional functions and constants used in this notebook
import sys, os
sys.path.append(os.path.abspath('..'))
from CommissFunctions import generate_task_plan 
from CommissFunctions import data_rate, bits_per_pix_VIS, compression_fractor_VIS, frame_time_VIS, stored_frames_per_int_VIS, pass_time_min


In [ ]:
''' Define a PandoraSat object for calling constants '''
p = ps.PandoraSat()

## Data Volume

Estimate data volume based on obseravtion parameters and estimated rates and sizes. Currently, we're storing data volume constants in CommissFunctions.py, but we my want to store those elsewhere where everyone can reference the most recent numbers.

In [ ]:
''' setting observing parameters for the task '''
VIS_xpix              = 200 ### saving three 200x200 ROIs
VIS_ypix              = 200 
regions               = 3  
frames_per_int        = 1
num_int               = 30000 

''' Calculating data volumes '''
int_and_reset_time    = frame_time_VIS * frames_per_int
bits_per_int          = VIS_xpix * VIS_ypix * regions * bits_per_pix_VIS * stored_frames_per_int_VIS
bits_per_sec          = bits_per_int / int_and_reset_time
test_time             = num_int * int_and_reset_time  ### seconds
bits_per_sec_comp     = bits_per_sec / compression_fractor_VIS
bits_test_total       = bits_per_sec_comp * test_time
Gbits_test_packet     = bits_test_total / 1E9 * 1.1 * 1.25
downlinks             = Gbits_test_packet * 1E9 / (data_rate * 1E6) /60 /pass_time_min 

print('test time:', test_time/60/60, 'hours')
print(Gbits_test_packet, 'Gbits')
print(downlinks, 'downlinks')


''' Test reducing temporal resolution for lower data volume. '''
### This task produces a high data volume. If we need to reduce it, we can reduce the temporal resolution. 
### The following lines compute how much reduction is needed to achieve the desired data volume.
reduce_by = 4 # Assuming a 5 min downlink data/15 minute observe but no downlink scheme
print('Reduced Temporal Resolution')
print(Gbits_test_packet/reduce_by, 'Gbits')
print(downlinks/reduce_by, 'downlinks')

## Define Targets

For now, we select an arbitrary primary target to place at the center of the frame. Two secondary targets are selected by PandoraSim and are the next two brightest objects in frame.

In [ ]:
### Possible Targets:
# Center: Gaia DR3 3143682556189212672 (118.985495, 5.825131)
# Right: Gaia DR3 3143682384390312704 (118.9134631658, 5.8164690058)
# Left: Gaia DR3 3143694096762670336 (119.0679737659, 5.8245399195)

In [ ]:
''' Finds target coordinates based on astropy '''
### You can do this differently as long as you have DEC and RA
target = "G3143682556189212672" # central target Gaia DR3 3143682556189212672
target2 = "G3143682384390312704" # in case we can use these in future to specify ROI regions (right most target) Gaia DR3 3143682384390312704
target3 = "G3143694096762670336" # in case we can use these in future to specify ROI regions (left most target) Gaia DR3 3143694096762670336

c = SkyCoord(ra=118.985495, dec=5.825131, unit='deg') # coordinates for "target" from GAIA database with more recent epoch then SkyCoord pulls from name
c2 = SkyCoord(ra=118.9134631658, dec=5.8164690058, unit='deg')
c3 = SkyCoord(ra=119.0679737659, dec=5.8245399195, unit='deg')

''' Identify the start time and end time of the observation '''
start_time = Time.now() #start time is now. This should get overridden by the scheduler. 
time_delta = TimeDelta(test_time, format='sec')
end_time = start_time + time_delta

## Create Simulated Observation

In [ ]:
''' Initialize a simulator object '''
### If there are not enough targets that meet Pandora's observable criteria, you may get an error. 
sim = VisibleSim(nROIs=regions, ROI_size=(VIS_xpix, VIS_ypix)) 


''' Point the simulator at the above target '''
###        RA,   DEC,    Theta 
sim.point(c.ra, c.dec, 0*u.deg) 


''' Identify the correct ROIs for our targets '''
### Currently, this is commented out and we're just taking the three default ROIs. Stay tuned for updates.
### Soon, a sim update will make it easier specify specific targets from the catalog of stars in frame. 
#sim.ROI_corners = [(924.0, 924.0), (1000, 1500), (500, 1000)]
print('ROI locations [pix]: ', sim.ROI_corners)


''' Save ROI locations in sky coordinates for later '''
### Once the above update is made, we may do this differently.
ROI0_coord = sim.pixel_to_world(sim.ROI_corners[0][0], sim.ROI_corners[0][1])
ROI1_coord = sim.pixel_to_world(sim.ROI_corners[1][0], sim.ROI_corners[1][1])
ROI2_coord = sim.pixel_to_world(sim.ROI_corners[2][0], sim.ROI_corners[2][1])

In [ ]:
''' sim has various attributes you can print out '''
#print(sim.ra, sim.dec, sim.wcs)

''' Notably we have a Catalog of objects visable in the VIS FOV '''
cat = sim.source_catalog
#print(cat)

In [ ]:
''' Plot the Full Frame Image. ROIs are in red boxes defining the subarray. '''
sim.show_FFI()

In [ ]:
''' Create an Observation '''

### This returns an astropy.io.fits.HDUList array. 
# `nreads` sets how many reads of the detector are coadded together to create a frame.
# `bin_frames` is a shortcut parameter to speed up computation. Must be a factor of nreads. We can ignore this for low nreads. 
# `nframes` sets how many frames will be returned.
# `start_time` indicates the time of observation. 
# `output_type` is set to array, for now. This make it easy to call, but we'll save it as a fits file later. 
###

''' This takes a really long time. For a quick look, reduce nframes by a reduce_factor.'''
reduce_factor = 1
data = sim.observe(nreads=frames_per_int, bin_frames=1, nframes=int(num_int/reduce_factor), start_time=start_time, output_type="array")

### The shape is: (num ROIs, num frames, nrows, ncolumns)
data.shape

In [ ]:
''' Plot our ROIs (regions of interest). ''' 
sim.show_ROI()

In [ ]:
''' Plot animations of ROIs frame-to-frame. Adjust index to switch ROIs. ''' 
index = 0
animate(data[index])

### If you want to save this:
#save_mp4(data[index], "3_12_animation.mp4")
#save_gif(data[index], "3_12_animation.gif")

## Save the simulated observation

This .fits file cannot be uploaded to GitHub! They can instead be saved on Box.

In [ ]:
''' Re-Create the observation in the correct format for the .fits file (i.e., output_type set to default) '''
### Before, object_type was array for easy computation in this notebook, but that is not correct for the .fits file
### NOTE: this still includes the reduce factor for faster computation. 

hdulist = sim.observe(nreads=frames_per_int, nframes=int(num_int/reduce_factor), bin_frames=1, start_time=start_time)


''' Save our simulated observation in a fits file '''
targ = target.replace(" ", "")
rois_outfile = "3.12_rois_"+str(targ)+".fits" ### REMEMBER to change the task number for 3.12 or 3.17
hdulist.writeto(rois_outfile, overwrite=True)

Show what is in our fits file: 

FITS file extensions: 
1. PRIMARY with some information about the simulated observation 
2. SCIENCE with information about the data
3. ROIs with information about each ROI 

In [ ]:
hdul_rois = fits.open(rois_outfile)
print(hdul_rois.info())

#print('.')
#print('.')
#print('.')
#print('-- PRIMARY Headers --')
#print(hdul_rois[0].header)

#print('.')
#print('.')
#print('.')
#print('-- SCIENCE Headers --')
#print(hdul_rois[1].header)

#print('.')
#print('.')
#print('.')
#print('-- ROI Headers --')
#print(hdul_rois[2].header)

## Data Product / Analysis Check

This commissioing task does not have a numbered data product. Instead, we include a "check."

### Check: The data should measure the pointing accuracy (absolute) and precision (rms variation from nominal) in arcseconds over time. Measure jitter from expected location. 

Right now, we aren't including much here. If useful, this section of the notebook could be used to perform and document analysis comparing the simulated observation to the true observation. 

In [ ]:
plt.imshow(hdul_rois['SCIENCE'].data[0], origin='lower', vmax=300)
plt.colorbar()
plt.title('Simulated image')

## Generate SOC File

The format of the SOC file is currently being developed. This function/variables will be updated accordingly.

In [ ]:
'''Here we define the information required for scheduling the observation.'''

variables = {
    'visit_id': '0312', ### setting based on task number, for now. Could change. REMEMBER to update for 3.12/3.17
    'obs_id': '000', ### this adjusts for tasks with multiple observations
    'target': target, 
    'priority': 1, 
    'start_time': start_time, 
    'stop_time': end_time, 
    'RA': c.ra.deg,
    'DEC': c.dec.deg,

    'NIR_AvgGroups': '', ### no NIR. 
    'NIR_ROI_StartX': '', 
    'NIR_ROI_StartY': '', 
    'NIR_ROI_SizeX': '', 
    'NIR_ROI_SizeY': '',
    'NIR_SC_Resets1': '',
    'NIR_SC_Resets2': '', 
    'NIR_SC_DropFrames1': '',
    'NIR_SC_DropFrames2': '',
    'NIR_SC_DropFrames3': '', 
    'NIR_SC_ReadFrames': '',
    'NIR_targetID': '', 
    'NIR_SC_Groups': '',
    'NIR_SC_Integrations': '',

    'ffi_flag': 0, # this task does not require full frame images
    'VIS_StarRoiDetMethod': 0,  ### 0: monitor stars at positions defined in [PredefinedStarRoiRa, PredefinedStarRoiDec]. 1: Run star detection algorithm on first image and obtain star ROIs from results.
    'VIS_FramesPerCoadd': frames_per_int, 
    'VIS_NumTotalFramesRequested': num_int*frames_per_int, 
    'VIS_TargetRA': c.ra.deg,
    'VIS_TargetDEC': c.dec.deg,
    'VIS_IncludeFieldSolnsInResp': 1, ### Determines whether the Ra/Dec/Rot values calculated for each frame in the visible science collect will be included in the response message [1] or not [0].
    'VIS_StarRoiDimension': [VIS_xpix, VIS_ypix], 
    'VIS_MaxNumStarRois': 0, ### max number of star ROIs that will be used for coadding signal from target stars if StarRoiDetMethod == 1.
    'VIS_numPredefinedStarRois': regions, ### If StarRoiDetMethod==0, this is the number of [RA, DEC] coordinates contained in the PredefinedStarRoiRA and PredefinedStarRoiDEC vectors.
    'VIS_PredefinedStarRoiRa': [c.ra.deg, c2.ra.deg, c3.ra.deg], ### currently defining as corner of ROI. We'll update this to be ROI centers.
    'VIS_PredefinedStarRoiDec': [c.dec.deg, c2.dec.deg, c3.dec.deg],
    'VIS_targetID': target, 
    'VIS_NumExposuresMax': num_int*frames_per_int, 
    'VIS_ExposureTime_us': frame_time_VIS * 1000000  
}

In [ ]:
'''Generate the SOC .xml file'''

output_soc_file = variables['visit_id']+'_'+variables['obs_id']+'_SOC.xml' 
generate_task_plan(variables, output_soc_file)